# Pipeline: Correction using Lapa LLM

This pipeline corrects zero-shot translations from OPUS-MT and mBART. To do that we ask Lapa LLM to rewrite target
sentences in way that would adhere to the source gender of a sentence.

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="lapa-llm/lapa-v0.1.2-instruct-GGUF",
	filename="lapa-v0.1.2-instruct-Q4_K_M.gguf",
)

In [ ]:
def ensure_desired_gender(sentence: str, target_gender: str):
    editing_uk_texts = [
        {"role": "system", "content": "You are a linguistic assistant for Ukrainian text editing."},
        {"role": "user", "content": (
            f"Edit the following Ukrainian sentence by ensuring the occupation is in the {target_gender} form, "
            f"keeping the structure of the sentence unchanged:\n{sentence}"
        )}
    ]
    output = llm.create_chat_completion(messages=editing_uk_texts)
    corrected_sentence = output['choices'][0]['message']['content']
    return corrected_sentence
    

ensure_desired_gender(sentence='Я -- чудовий механік', target_gender='female')

## mBART

In [ ]:
import pandas as pd
from tqdm import tqdm


tqdm.pandas()

df = pd.read_csv('../data/mbart.zeroshot.translations.tsv', index_col=None, sep='\t')
df['mbart_llm_corrected'] = df.progress_apply(lambda r: ensure_desired_gender(r['mbart_hypothesis'], r['source_gender']), axis=1)
df.head()

In [6]:
df.to_csv("../data/mbart.llm_corrected.translations.tsv", sep="\t", index=False)

## OPUS-MT

In [ ]:
df = pd.read_csv('../data/opusmt.zeroshot.translations.tsv', index_col='id', sep='\t')
df['opusmt_llm_corrected'] = df.progress_apply(lambda r: ensure_desired_gender(r['opusmt_hypothesis'], r['source_gender']), axis=1)
df.head()
df.to_csv("../data/opusmt.llm_corrected.translations.tsv", sep="\t", index=False)
